# Tutorial 8: Penalty and Augmented Lagrangian Methods

**Prerequisites:** Tutorial 5 — Newton and Quasi-Newton Methods, Tutorial 6 — KKT Conditions, Tutorial 7 — Sequential Quadratic Programming (SQP)  
**Up Next:** Tutorial 9

---

## Concept

SQP solves constrained problems elegantly but requires a specialized solver that handles constraints natively. **Penalty methods** take a different approach: transform the constrained problem into a sequence of *unconstrained* problems by adding a penalty term that grows when constraints are violated. Any unconstrained optimizer (BFGS, gradient descent, etc.) can then be applied directly.

The simplest variant is the **exterior quadratic penalty**: $\phi(\mathbf{x}; \mu) = f(\mathbf{x}) + \mu \sum_i [\max(0, g_i(\mathbf{x}))]^2$. As $\mu \to \infty$, the minimizer of $\phi$ approaches the constrained optimum. The **augmented Lagrangian** improves on this by also estimating Lagrange multipliers, achieving exact convergence with finite $\mu$.

## Key Takeaway

> **Penalty methods convert constraints into objective-function penalties, letting you use any unconstrained optimizer. Increasing the penalty weight drives the solution toward feasibility, but large weights cause ill-conditioning. The augmented Lagrangian avoids this by incorporating multiplier estimates.**

## Math Core

**Exterior quadratic penalty:**

$$\phi(\mathbf{x}; \mu) = f(\mathbf{x}) + \mu \sum_i \left[\max\left(0,\; g_i(\mathbf{x})\right)\right]^2$$

Solve $\min_\mathbf{x} \phi(\mathbf{x}; \mu_k)$ for increasing $\mu_1 < \mu_2 < \cdots$. Each subproblem is unconstrained.

**Augmented Lagrangian:**

$$\mathcal{L}_A(\mathbf{x}, \lambda; \mu) = f(\mathbf{x}) + \sum_i \left[ \lambda_i \, g_i(\mathbf{x}) + \frac{\mu}{2} \, [\max(0, g_i(\mathbf{x}))]^2 \right]$$

After each minimization, update $\lambda_i \leftarrow \lambda_i + \mu \, \max(0, g_i(\mathbf{x}))$. This allows convergence to the exact constrained optimum without sending $\mu \to \infty$.

## Code

We implement the exterior penalty method for our four-bar problem with the Grashof constraint, then compare against the SLSQP solution from Tutorial 7.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
from scipy.optimize import minimize
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])


def grashof_g(x):
    """Grashof constraint: g(x) <= 0 means satisfied."""
    lengths = np.array([L0, L1, x[0], x[1]])
    s = np.sum(lengths)
    return (np.max(lengths) + np.min(lengths)) - (s - np.max(lengths) - np.min(lengths))

### Exterior quadratic penalty

We solve a sequence of unconstrained problems with increasing penalty weight $\mu$.

In [ ]:
def penalized_objective(x, mu):
    """f(x) + mu * max(0, g(x))^2"""
    gv = grashof_g(x)
    return objective(x) + mu * max(0.0, gv)**2

x0 = np.array([1.0, 1.5])
bnds = [(0.3, 3.0), (0.3, 3.0)]
mu_values = [1, 10, 100, 500, 1000, 5000]

penalty_results = []
x_warm = x0.copy()
for mu in mu_values:
    res = minimize(lambda x, _mu=mu: penalized_objective(x, _mu),
                   x_warm, method='L-BFGS-B', bounds=bnds,
                   options={'ftol': 1e-12, 'eps': 1e-7})
    gv = grashof_g(res.x)
    penalty_results.append((mu, res.x.copy(), objective(res.x), gv))
    x_warm = res.x.copy()  # warm-start next subproblem
    print(f'mu={mu:5d}: l2={res.x[0]:.4f}, l3={res.x[1]:.4f}, '
          f'f={objective(res.x):.4f}, g={gv:.4f}')

### Effect of increasing $\mu$

As $\mu$ grows, the constraint violation shrinks toward zero, while the objective value increases (the constraint becomes harder to violate).

In [ ]:
mus = [r[0] for r in penalty_results]
fs = [r[2] for r in penalty_results]
gs = [r[3] for r in penalty_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.semilogx(mus, fs, 'bo-', ms=6, lw=2)
ax1.set_xlabel(r'Penalty weight $\mu$')
ax1.set_ylabel(r'Objective $f(\mathbf{x}^*)$')
ax1.set_title('Objective at penalty solution')
ax1.grid(True, alpha=0.3)

ax2.semilogx(mus, gs, 'rs-', ms=6, lw=2)
ax2.axhline(0, color='k', ls='--', lw=1)
ax2.set_xlabel(r'Penalty weight $\mu$')
ax2.set_ylabel(r'Constraint $g(\mathbf{x}^*)$')
ax2.set_title('Constraint violation at penalty solution')
ax2.grid(True, alpha=0.3)
plt.tight_layout()

### Comparison with SLSQP

Let's compare the penalty-method solutions against the SLSQP result from Tutorial 7.

In [ ]:
# SLSQP reference
res_sqp = minimize(objective, x0, method='SLSQP', bounds=bnds,
                   constraints={'type': 'ineq', 'fun': lambda x: -grashof_g(x)})

print(f'SLSQP:         l2={res_sqp.x[0]:.4f}, l3={res_sqp.x[1]:.4f}, '
      f'f={res_sqp.fun:.4f}, g={grashof_g(res_sqp.x):.4f}')
print(f'Penalty (best): l2={penalty_results[-1][1][0]:.4f}, '
      f'l3={penalty_results[-1][1][1]:.4f}, '
      f'f={penalty_results[-1][2]:.4f}, g={penalty_results[-1][3]:.4f}')

### Trajectories on the landscape

Let's see where the penalty solutions land relative to SLSQP and the Grashof boundary.

In [ ]:
l2v = np.linspace(0.3, 2.5, 80)
l3v = np.linspace(0.3, 2.5, 80)
L2, L3 = np.meshgrid(l2v, l3v)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)
G = np.vectorize(lambda a, b: grashof_g([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
ax.contourf(L2, L3, G, levels=[0, 10], colors=['red'], alpha=0.2)
ax.contour(L2, L3, G, levels=[0], colors=['red'],
           linewidths=2, linestyles='--')
# Penalty solutions path
pen_xs = np.array([r[1] for r in penalty_results])
ax.plot(pen_xs[:, 0], pen_xs[:, 1], 'wo-', ms=6, lw=2,
        label='Penalty (increasing $\\mu$)')
for i, (mu, _, f, _) in enumerate(penalty_results):
    ax.annotate(f'$\\mu$={mu}', pen_xs[i], fontsize=7,
                color='white', ha='left')
# SLSQP solution
ax.plot(*res_sqp.x, 'r*', ms=14, markeredgecolor='k',
        label=f'SLSQP (f={res_sqp.fun:.3f})')
ax.plot(*x0, 'ko', ms=8, label='start')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Penalty solutions converge toward feasible boundary')
ax.legend(loc='upper left', fontsize=8)
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

Notice that the penalty solutions approach the Grashof boundary as $\mu$ increases but may converge to a different point on it than SLSQP. This is because the penalty landscape has different local minima than the original constrained problem — a well-known limitation of penalty methods.

The augmented Lagrangian improves on this by incorporating Lagrange multiplier estimates, but the fundamental trade-off remains: penalty methods are simple and widely applicable, while SQP is more precise for smooth problems.

## Mechanism Hook

In practice, penalty methods shine when constraints are complex, non-smooth, or come from simulation (black-box constraints). For the four-bar linkage, we might have additional constraints beyond Grashof — link length limits, workspace bounds, transmission angle requirements — that are hard to linearize for SQP. A penalty approach lets us pile on arbitrary constraints without changing the optimizer.

In [ ]:
from dms.mechanisms.fourbar import FourBar

# Show best penalty solution
x_pen = penalty_results[-1][1]
l2, l3 = x_pen
mechanism = FourBar(L0, L1, l2, l3)
path = np.array([coupler_point(t, l2, l3) for t in THETAS])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Penalty result ($\\mu$={penalty_results[-1][0]}): '
             f'$\\ell_2$={l2:.3f}, $\\ell_3$={l3:.3f}\n'
             f'f={objective(x_pen):.4f}, g={grashof_g(x_pen):.4f}')
plt.tight_layout()

We now have a full toolkit for constrained optimization: KKT theory (Tutorial 6), SQP for smooth problems (Tutorial 7), and penalty methods for everything else. The next tutorials will explore global and population-based methods that escape local minima entirely.

---